<a href="https://colab.research.google.com/github/thanusree02/Natural-Language-Processing/blob/main/NLP_LAB_14_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Import Libraries

In [46]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd

In [47]:
texts = [
    "free money now",
    "call now free offer",
    "hello how are you",
    "let us meet tomorrow",
    "win cash prize",
    "are you available tomorrow"
]

labels = [1, 1, 0, 0, 1, 0]

In [48]:


# Step 3: Convert text to BoW
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(texts).toarray()
y = labels

In [49]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)


In [50]:
# Step 5: Create Dataset class
class TextDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

In [51]:
train_loader = DataLoader(TextDataset(X_train, y_train), batch_size=2, shuffle=True)
test_loader = DataLoader(TextDataset(X_test, y_test), batch_size=2)


CNN Model (1D CNN for Text)

In [52]:
class CNNModel(nn.Module):
    def __init__(self, input_size):
        super().__init__()

        kernels=4
        kernel_size=3
        self.conv = nn.Conv1d(1, kernels, kernel_size)

        pool_size=2
        self.pool = nn.MaxPool1d(pool_size)

        # Calculate flattened size
        flattened_size = kernels * ((input_size - pool_size)//2)

        # Hidden layer
        hidden_neurons=8
        self.fc1 = nn.Linear(flattened_size, hidden_neurons)   # 8 neurons (you can change)

        # Output layer
        output_neurons=1
        self.fc2 = nn.Linear(hidden_neurons, output_neurons)

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = x.unsqueeze(1)              # (batch, 1, features)
        x = torch.relu(self.conv(x))
        x = self.pool(x)

        x = x.view(x.size(0), -1)       # Flatten

        x = torch.relu(self.fc1(x))     # Hidden layer + activation
        x = self.fc2(x)                 # Output layer

        return self.sigmoid(x)

In [53]:
model = CNNModel(input_size=X_train.shape[1])

In [54]:
# Step 7: Loss and Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [55]:
# Step 8: Training
for epoch in range(10):
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # Convert X_batch to float before passing to the model
        output = model(X_batch.float()).squeeze()
        # Convert y_batch to float for BCELoss
        loss = criterion(output, y_batch.float())
        loss.backward()
        optimizer.step()
    print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 1, Loss: 0.7265
Epoch 2, Loss: 0.9295
Epoch 3, Loss: 0.6966
Epoch 4, Loss: 0.5064
Epoch 5, Loss: 0.6677
Epoch 6, Loss: 0.6598
Epoch 7, Loss: 0.6272
Epoch 8, Loss: 0.4717
Epoch 9, Loss: 0.6122
Epoch 10, Loss: 0.5752


In [56]:
y_pred = []
y_actual = []

model.eval()
with torch.no_grad():
    for X_batch, y_batch in test_loader:
        output = model(X_batch).squeeze()
        preds = (output > 0.5).int()
        y_pred.extend(preds.tolist())
        y_actual.extend(y_batch.tolist())


# Convert actual_labels from float to int if needed by sklearn metrics
y_actual = [int(y) for y in y_actual]
print(y_pred)

[1, 0]


In [57]:
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

print("\nConfusion Matrix:")
print(confusion_matrix(y_actual, y_pred))

print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

print("\nAccuracy Score:", accuracy_score(y_actual, y_pred))



Confusion Matrix:
[[1 0]
 [0 1]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00         1
           1       1.00      1.00      1.00         1

    accuracy                           1.00         2
   macro avg       1.00      1.00      1.00         2
weighted avg       1.00      1.00      1.00         2


Accuracy Score: 1.0
